# 附录：用最少代码训练一个“简单聊天模型”

这个小实验的目标很简单：

```text
输入：你好
输出：你好呀，很高兴见到你！
```

它不是 GPT-2 本体，但思想和 GPT-2 很像：**根据前面的文字，预测下一个字，不断预测下去，就生成了一句话。**

我们故意把模型做得很小，只训练它学会几种简单聊天，方便课堂演示。

## 1. 准备一小批训练数据

如果第一次运行停在这个代码块几秒钟，通常是在启动 Jupyter kernel 或加载 PyTorch，不是代码卡死。

格式统一成：

```text
用户：问题
助手：回答<结束>
```

模型会学习这种对话文本的规律。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print('PyTorch 已加载，开始准备训练数据。')

pairs = [
    ('你好', '你好呀，很高兴见到你！'),
    ('hello', 'Hi，很高兴见到你！'),
    ('hi', 'Hi，很高兴和你聊天。'),
    ('早上好', '早上好，祝你今天学习顺利！'),
    ('晚上好', '晚上好，今天辛苦了。'),
    ('在吗', '在的，有什么想问我？'),
    ('晚安', '晚安，祝你休息好。'),
    ('谢谢', '不客气，很高兴帮到你。'),
    ('再见', '再见，下次继续学习。'),

    ('你是谁', '我是一个很小的课堂聊天模型。'),
    ('你叫什么', '你可以叫我 TinyGPT。'),
    ('你能做什么', '我可以演示一个小模型如何根据输入生成回复。'),
    ('你会什么', '我会回答一些训练过的简单问题。'),
    ('你聪明吗', '我很小，只能学会训练数据里的简单规律。'),

    ('什么是 AI', 'AI 可以理解成一个会学习的函数：输入信息，输出结果。'),
    ('什么是模型', '模型就是从数据中学到的函数。'),
    ('什么是训练', '训练就是不断调整参数，让模型输出更接近答案。'),
    ('什么是大模型', '大模型是用更多数据和参数训练出来的更通用模型。'),
    ('什么是 token', 'token 是模型处理文本时使用的基本单位。'),

    ('我想学习 AI', '可以先理解输入、模型、输出，再学习机器学习和神经网络。'),
    ('怎么学 AI', '先学 Python 和机器学习基础，再学深度学习和大模型应用。'),
    ('机器学习难吗', '刚开始会有点抽象，但用图和小实验会容易很多。'),
    ('深度学习是什么', '深度学习是用多层神经网络从数据中学习规律。'),
    ('今天学什么', '今天先学 AI 的整体架构和最小聊天模型。'),
]

text = '\n'.join([f'用户：{q}\n助手：{a}<结束>' for q, a in pairs])
text = '\n'.join([text] * 8)  # 小数据重复几遍，帮助小模型记牢课堂样本
print(text[:160])

PyTorch 已加载，开始准备训练数据。
用户：你好
助手：你好呀，很高兴见到你！<结束>
用户：hello
助手：Hi，很高兴见到你！<结束>
用户：hi
助手：Hi，很高兴和你聊天。<结束>
用户：早上好
助手：早上好，祝你今天学习顺利！<结束>
用户：晚上好
助手：晚上好，今天辛苦了。<结束>
用户：在吗
助手：在的，有什么想问我？<结束>
用户：晚安



## 2. 把文字变成数字

神经网络不能直接吃文字，所以先把每个字符变成编号。

In [2]:
chars = sorted(set(text)) + ['<未知>']
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
unk_id = stoi['<未知>']

def encode(s):
    return [stoi.get(ch, unk_id) for ch in s]

def decode(ids):
    return ''.join(itos[i] for i in ids if itos[i] != '<未知>')

data = torch.tensor(encode(text), dtype=torch.long)
vocab_size = len(chars)
block_size = 64

print('字符表大小:', vocab_size)
print('训练文本长度:', len(data))

字符表大小: 178
训练文本长度: 6407


## 3. 构造训练批次

训练任务是：看到前面一串字符，预测下一个字符。

In [3]:
def get_batch(batch_size=32):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch(2)
print('输入形状:', x.shape)
print('目标形状:', y.shape)

输入形状: torch.Size([2, 64])
目标形状: torch.Size([2, 64])


## 4. 定义一个很小的 GPT-like 模型

GPT-2 的核心是 Transformer。这里我们只用两层很小的 Transformer，让它学会简单聊天。

In [4]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_emb=64, n_head=4, n_layer=2):
        super().__init__()
        self.token = nn.Embedding(vocab_size, n_emb)
        self.position = nn.Embedding(block_size, n_emb)
        layer = nn.TransformerEncoderLayer(n_emb, n_head, 4*n_emb, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layer)
        self.head = nn.Linear(n_emb, vocab_size)

    def forward(self, x):
        B, T = x.shape
        h = self.token(x) + self.position(torch.arange(T))
        mask = torch.triu(torch.ones(T, T) * float('-inf'), diagonal=1)
        h = self.transformer(h, mask)
        return self.head(h)

model = TinyGPT(vocab_size)
print('参数量:', sum(p.numel() for p in model.parameters()))

参数量: 127026


## 5. 开始训练

损失越低，说明模型越会预测下一个字。

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

for step in range(1201):
    xb, yb = get_batch(batch_size=32)
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 300 == 0:
        print(step, round(loss.item(), 4))

0 5.3639


300 0.0832


600 0.0616


900 0.0582


1200 0.0481


## 6. 让模型回复一句话

生成时，只给它：

```text
用户：你好
助手：
```

然后让模型一个字一个字往下写，直到写到 `<结束>`。

In [6]:
@torch.no_grad()
def chat(message, max_new_tokens=60):
    prompt = f'用户：{message}\n助手：'
    idx = torch.tensor([encode(prompt)], dtype=torch.long)

    for _ in range(max_new_tokens):
        logits = model(idx[:, -block_size:])
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.argmax(probs, dim=-1, keepdim=True)
        idx = torch.cat([idx, next_id], dim=1)

        out = decode(idx[0].tolist())
        if '<结束>' in out:
            break

    return out.split('助手：')[-1].split('<结束>')[0]

for q in ['你好', '你是谁', '你能做什么', '什么是 AI', '怎么学 AI', '再见']:
    print(q, '=>', chat(q))

你好 => 你好呀，很高兴见到你！
你是谁 => 我是一个很小的课堂聊天模型。
你能做什么 => 我可以演示一个小模型如何根据输入生成回复。
什么是 AI => AI 可以理解成一个会学习的函数：输入信息，输出结果。
怎么学 AI => 先学 Python 和机器学习基础，再学深度学习和大模型应用。
再见 => 再见，下次继续学习。


## 7. 这个小模型说明了什么

它很小，只会回答训练数据附近的简单问题，但它已经具备大模型最核心的生成逻辑：

```text
上下文 → 预测下一个字 → 接着预测 → 形成回复
```

真正的 GPT-2 / GPT-4 区别在于：

- 数据大得多。
- 模型大得多。
- 训练时间长得多。
- 上下文更长。
- 学到的语言和知识更丰富。

但第一性原理是一样的：**模型不是提前写死回答，而是根据上下文一步步生成。**